# Import necessary libraries

In [1]:
!pip install gdown --q
!pip install pydrive --q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 987.4/987.4 kB 22.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import os
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data as data
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torchvision import models, transforms
from torchvision.transforms import ToTensor
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import gdown
import zipfile
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials
from torch.optim.lr_scheduler import ReduceLROnPlateau
import copy

In [3]:
# Set device for training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Section 1: Dataset loading and preprocessing

In [4]:
# Set up Google Drive authentication
def authenticate_google_drive():
    """Authenticate the user for Google Drive access."""
    auth.authenticate_user()
    gauth = GoogleAuth()
    # Get the credentials authenticated by google.colab.auth
    gauth.credentials = GoogleCredentials.get_application_default()
    # Explicitly authorize PyDrive with the obtained credentials
    gauth.Authorize()
    return GoogleDrive(gauth)

In [5]:
# Download function for CSV files
def download_csv_files(drive, folder_id, output_dir='csv_files'):
    """Download all CSV files from a Google Drive folder."""
    os.makedirs(output_dir, exist_ok=True)
    file_list = drive.ListFile({'q': f"'{folder_id}' in parents and trashed=false"}).GetList()

    for file in file_list:
        if file['title'].endswith('.csv'):
            print(f"Downloading {file['title']}...")
            file.GetContentFile(f'{output_dir}/{file["title"]}')

In [6]:
# Download and extract zip file from Google Drive
def download_and_extract_zip(file_id, output_path, extract_to):
    """Download and extract dataset from Google Drive."""
    download_url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(download_url, output_path, quiet=False)

    with zipfile.ZipFile(output_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

In [7]:
import os
import pandas as pd

def prepare_dataframe(csv_path, image_dir):
    """Prepare a DataFrame with image paths and labels, robustly matching image IDs to filenames."""
    df = pd.read_csv(csv_path)

    # Get all actual image filenames in the directory and create a map for robust lookup
    actual_image_files = {}
    if os.path.exists(image_dir):
        for filename in os.listdir(image_dir):
            base_name, ext = os.path.splitext(filename)
            if ext.lower() in ('.jpg', '.jpeg', '.png'): # Only consider common image file extensions
                actual_image_files[base_name] = filename
    else:
        print(f"Warning: Image directory {image_dir} does not exist.")
        return pd.DataFrame() # Return empty DataFrame if dir doesn't exist

    # --- Diagnostic prints start here ---
    print(f"\nDebugging matching for {csv_path} with images in {image_dir}")
    if not df.empty:
        print(f"Sample IDs from CSV (first 5): {df['ID'].head().tolist()}")
    if actual_image_files:
        # Convert keys to list for sampling, handle potential errors if actual_image_files is empty
        sample_keys = list(actual_image_files.keys())
        print(f"Sample image base names from directory (first 5): {sample_keys[:5]}")
    # --- Diagnostic prints end here ---

    matched_data = []
    for index, row in df.iterrows():
        image_id = str(row['ID']) # Ensure ID is string for lookup
        if image_id in actual_image_files:
            full_path = os.path.join(image_dir, actual_image_files[image_id])
            # Create a dictionary for the current row, including the matched image_path
            row_data = row.to_dict()
            row_data['image_path'] = full_path
            matched_data.append(row_data)

    # Create a new DataFrame from the matched data
    df = pd.DataFrame(matched_data)

    print(f"Total matched images: {len(df)}")
    return df

In [8]:
# Split CUIs (Concept Unique Identifiers) for multi-label classification
def split_cuis(df, column_name='CUIs'):
    """Split CUIs into lists for multi-label classification."""
    df[column_name] = df[column_name].apply(lambda x: x.split(';'))
    return df

In [9]:
class MedicalImageDataset(Dataset):
    def __init__(self, df, labels, transform=None):
        self.df = df
        self.labels = labels  # These should be the binary labels of shape [num_samples, num_classes]
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['image_path']
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.float32)

In [ ]:
# Main function for data preparation

def download_data(csv_zip_file_id):
    """Download the dataset from Google Drive."""

    # Using gdown for CSV zip file
    csv_zip_output_path = 'csv_files.zip'
    csv_extract_to = 'csv_files'
    os.makedirs(csv_extract_to, exist_ok=True) # Ensure directory exists for extraction
    download_and_extract_zip(csv_zip_file_id, csv_zip_output_path, csv_extract_to)

    # Download and extract train and validation image datasets
    train_image_file_id = 'Your Drive Folder ID'
    download_and_extract_zip(train_image_file_id, 'croppedImages.zip', 'train_images_folder')

    valid_image_file_id = 'Your Drive Folder ID'
    download_and_extract_zip(valid_image_file_id, 'valid_cropped_images.zip', 'valid_images_folder')

# The call to download_data() will be placed in a new cell after the user provides the csv_zip_file_id

In [ ]:
# Extract file ID from the provided Google Drive link
csv_zip_file_id = 'Your Drive Folder ID'

# Call the download_data function with the extracted file ID
download_data(csv_zip_file_id)

In [12]:
def find_image_directory(base_dir):
    """Recursively find the directory containing the most image files."""
    best_image_dir = base_dir
    max_images = 0

    if not os.path.exists(base_dir):
        print(f"Error: Base directory {base_dir} does not exist.")
        return base_dir # Or raise an error, depending on desired error handling

    print(f"Starting image directory search in: {base_dir}")
    for root, dirs, files in os.walk(base_dir):
        current_image_count = 0
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                current_image_count += 1
        # Diagnostic print to see if any images are found in the current root
        if current_image_count > 0:
            print(f"  In {root}, found {current_image_count} image files.")

        if current_image_count > max_images:
            max_images = current_image_count
            best_image_dir = root

    if max_images > 0:
        print(f"Found best image directory: {best_image_dir} with {max_images} images.")
        return best_image_dir
    else:
        print(f"Warning: No image files found in {base_dir} or its subdirectories.")
        # More diagnostics if no images found at all
        if os.path.exists(base_dir):
            if len(os.listdir(base_dir)) == 0:
                print(f"  {base_dir} appears to be an empty directory.")
            else:
                print(f"  {base_dir} contains items, but no recognized image files were found in any subdirectories during the scan.")
                print(f"  First 5 items in {base_dir}: {os.listdir(base_dir)[:5]}")
        return base_dir # Fallback if no images are found at all

def prepare_data():
    """Prepare the dataset for training and validation."""
    # Adjust image_dir to account for potential nested directory after extraction
    train_image_base_dir = '/content/train_images_folder'
    valid_image_base_dir = '/content/valid_images_folder'

    # Use the new helper function to find the actual image directories
    train_image_actual_dir = find_image_directory(train_image_base_dir)
    valid_image_actual_dir = find_image_directory(valid_image_base_dir)

    # Ensure directories are found and not empty
    if not os.path.exists(train_image_actual_dir) or not os.listdir(train_image_actual_dir):
        raise ValueError(f"Training image directory '{train_image_actual_dir}' is empty or not found. Check extraction process.")
    if not os.path.exists(valid_image_actual_dir) or not os.listdir(valid_image_actual_dir):
        raise ValueError(f"Validation image directory '{valid_image_actual_dir}' is empty or not found. Check extraction process.")

    # Prepare dataframes with image paths
    print(f"Using train image directory: {train_image_actual_dir}")
    train_df = prepare_dataframe('csv_files/train_concepts.csv', train_image_actual_dir)

    print(f"Using valid image directory: {valid_image_actual_dir}")
    valid_df = prepare_dataframe('csv_files/valid_concepts.csv', valid_image_actual_dir)

    # Check if dataframes are empty after image path matching
    if train_df.empty:
        raise ValueError("Training DataFrame is empty after matching images. Check image paths or CSV content.")
    if valid_df.empty:
        raise ValueError("Validation DataFrame is empty after matching images. Check image paths or CSV content.")

    # Split CUIs for multi-label classification
    train_df = split_cuis(train_df.copy()) # Use .copy() to avoid SettingWithCopyWarning
    valid_df = split_cuis(valid_df.copy()) # Use .copy() to avoid SettingWithCopyWarning

    # Initialize MultiLabelBinarizer and transform CUIs
    mlb = MultiLabelBinarizer()
    y_train = mlb.fit_transform(train_df['CUIs'])
    y_valid = mlb.transform(valid_df['CUIs']) # Use transform, not fit_transform for validation set

    print(f"Number of unique CUIs: {len(mlb.classes_)}")

    # Reduce image size to 128x128 for lower memory usage
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor()
    ])

    # Create datasets
    train_dataset = MedicalImageDataset(train_df, y_train, transform=transform)
    val_dataset = MedicalImageDataset(valid_df, y_valid, transform=transform)

    # Create DataLoaders
    batch_size = 128
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    print(f"Train size: {len(train_dataset)}")
    print(f"Validation size: {len(val_dataset)}")

    return train_loader, val_loader, mlb

In [13]:
train_loader, val_loader, mlb = prepare_data()

Starting image directory search in: /content/train_images_folder
  In /content/train_images_folder/train, found 59958 image files.
Found best image directory: /content/train_images_folder/train with 59958 images.
Starting image directory search in: /content/valid_images_folder
  In /content/valid_images_folder/valid, found 9904 image files.
Found best image directory: /content/valid_images_folder/valid with 9904 images.
Using train image directory: /content/train_images_folder/train

Debugging matching for csv_files/train_concepts.csv with images in /content/train_images_folder/train
Sample IDs from CSV (first 5): ['ROCOv2_2023_train_000001', 'ROCOv2_2023_train_000002', 'ROCOv2_2023_train_000003', 'ROCOv2_2023_train_000004', 'ROCOv2_2023_train_000005']
Sample image base names from directory (first 5): ['ROCOv2_2023_train_043093', 'ROCOv2_2023_train_038743', 'ROCOv2_2023_train_032565', 'ROCOv2_2023_train_021035', 'ROCOv2_2023_train_036426']
Total matched images: 59958
Using valid image 

# Section 3: Model setup

In [14]:
num_classes = len(mlb.classes_)  # Correct the number of classes based on MultiLabelBinarizer
def build_model(num_classes):
    """Build the lightest DenseNet model (DenseNet-201)."""
    model = models.densenet201(pretrained=True)
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)  # Adjust the classifier to match the number of output classes
    return model.to(device)

# Section 4: Training and evaluation functions

In [15]:
# Training and evaluation functions
def train_epoch(model, dataloader, criterion, optimizer):
    """Train the model for one epoch."""
    model.train()
    running_loss = 0.0
    for inputs, labels in tqdm(dataloader, desc='Training'):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()

        optimizer.step()
        running_loss += loss.item()

    return running_loss / len(dataloader)

In [16]:
def evaluate(model, dataloader, criterion):
    """Evaluate the model on the validation set."""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc='Evaluating'):
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            # Apply sigmoid to get probabilities and threshold at 0.5 for multi-label classification
            preds = torch.sigmoid(outputs) > 0.5  # Binary classification for each label
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    # Flatten the predictions and labels
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # Calculate metrics for multi-label classification using micro average
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='micro')
    recall = recall_score(all_labels, all_preds, average='micro')
    f1 = f1_score(all_labels, all_preds, average='micro')

    return running_loss / len(dataloader), accuracy, precision, recall, f1

# Section 5: Main training loop

In [17]:
# Build the model
model = build_model(num_classes)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet201_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet201_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/densenet201-c1103571.pth" to /root/.cache/torch/hub/checkpoints/densenet201-c1103571.pth


100%|██████████| 77.4M/77.4M [00:00<00:00, 179MB/s]


In [18]:
# Define loss function and optimizer for DenseNet
criterion = nn.BCEWithLogitsLoss()  # BCEWithLogitsLoss is used for multi-label classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [25]:
import os
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm  # For progress bar
import gdown
from torch.serialization import add_safe_globals

# Allow numpy reconstruct for safe unpickling
add_safe_globals([("numpy.core.multiarray", "_reconstruct")])

# Define the model name and save paths for a fresh start
model_name = 'DenseNet201_FreshStart_MultiLabel'
local_save_path = f"{model_name}.pth"
# This will be the canonical save path for the model in Google Drive
drive_save_path = f"/content/drive/MyDrive/Dense201/{local_save_path}"


In [28]:
def resume_training(model, optimizer, scheduler, train_loader, val_loader, criterion, mlb, drive_save_path, start_epoch=0, resume_epochs=10, best_f1=0.0, device="cuda"):
    """Resume training loop that saves the best model based on F1 score."""
    total_epochs = start_epoch + resume_epochs

    for epoch in range(start_epoch, total_epochs):
        print(f"\n🗓️ Epoch {epoch+1}/{total_epochs}")

        # Training with tqdm for progress bar
        model.train()
        train_loss = 0.0
        train_batches = len(train_loader)

        with tqdm(train_loader, desc=f"🛠️ Training Epoch {epoch+1}", unit="batch") as pbar:
            for batch in pbar:
                # Properly unpack the batch
                inputs, labels = batch[0], batch[1]
                inputs, labels = inputs.to(device), labels.to(device)

                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                # Update loss and progress bar
                train_loss += loss.item()
                pbar.set_postfix(loss=loss.item())

        train_loss /= train_batches

        # Validation
        val_loss, accuracy, precision, recall, f1 = evaluate(model, val_loader, criterion)

        print(f"📊 Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Acc: {accuracy:.4f} | Prec: {precision:.4f} | Rec: {recall:.4f} | F1: {f1:.4f}")

        # Step the scheduler
        scheduler.step(val_loss)

        # Save the model if F1 improved
        if f1 > best_f1:
            best_f1 = f1
            save_dict = {
                'model_state_dict': model.state_dict(),
                'input_size': (3, 128, 128),
                'num_classes': model.classifier.out_features,
                'class_names': mlb.classes_,
                'epoch': epoch + 1,
                'optimizer_state_dict': optimizer.state_dict(),
                'best_f1': best_f1
            }
            torch.save(save_dict, drive_save_path)
            print(f"✅ Best Model (F1: {best_f1:.4f}) saved to {drive_save_path}")
        else:
            print(f"ℹ️ Current F1 ({f1:.4f}) did not improve over best F1 ({best_f1:.4f}). Not saving model.")


    print("🎯 Training complete.")


In [29]:
# Initialize model, optimizer, and scheduler for a fresh start
print("🆕 Starting training from scratch with new model name.")
model = build_model(num_classes)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.2, patience=3, threshold=1e-3, cooldown=2, min_lr=1e-6)
best_f1 = 0.0
start_epoch = 0


🆕 Starting training from scratch with new model name.


In [30]:
import os

drive_save_dir = "/content/drive/MyDrive/Dense201/"
os.makedirs(drive_save_dir, exist_ok=True)

In [ ]:

# 🏋️‍♂️ Start the training process
resume_training(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    mlb=mlb,
    drive_save_path=drive_save_path,
    start_epoch=start_epoch,
    resume_epochs=15,
    best_f1=best_f1,
    device=device
)



🗓️ Epoch 1/15


Evaluating: 100%|██████████| 78/78 [01:19<00:00,  1.02s/it]


📊 Train Loss: 0.0110 | Val Loss: 0.0064 | Acc: 0.1334 | Prec: 0.8776 | Rec: 0.3737 | F1: 0.5242
✅ Best Model (F1: 0.5242) saved to /content/drive/MyDrive/Dense201/DenseNet201_FreshStart_MultiLabel.pth

🗓️ Epoch 2/15


Evaluating: 100%|██████████| 78/78 [01:21<00:00,  1.04s/it]


📊 Train Loss: 0.0059 | Val Loss: 0.0062 | Acc: 0.1365 | Prec: 0.8905 | Rec: 0.3681 | F1: 0.5208
ℹ️ Current F1 (0.5208) did not improve over best F1 (0.5242). Not saving model.

🗓️ Epoch 3/15


...

Evaluating: 100%|██████████| 78/78 [01:17<00:00,  1.00it/s]


📊 Train Loss: 0.0050 | Val Loss: 0.0060 | Acc: 0.1385 | Prec: 0.8778 | Rec: 0.3868 | F1: 0.5370
✅ Best Model (F1: 0.5370) saved to /content/drive/MyDrive/Dense201/DenseNet201_FreshStart_MultiLabel.pth

🗓️ Epoch 7/15


Evaluating: 100%|██████████| 78/78 [01:17<00:00,  1.00it/s]


📊 Train Loss: 0.0048 | Val Loss: 0.0062 | Acc: 0.1360 | Prec: 0.8623 | Rec: 0.3778 | F1: 0.5254
ℹ️ Current F1 (0.5254) did not improve over best F1 (0.5370). Not saving model.

🗓️ Epoch 8/15


🛠️ Training Epoch 8:  92%|█████████▏| 430/469 [10:29<00:56,  1.46s/batch, loss=0.00438]

In [ ]:
drive_save_path = "/content/drive/MyDrive/DenseNet201_FreshStart_MultiLabel_Decreasing_LR.pth"
print(os.path.exists(drive_save_path))  # Should return True if the path is correct


True
